In [0]:
%sql
-- Setting database target workspace explicitly
USE CATALOG retailmart_catalog;
USE SCHEMA operational_lakehouse;

In [0]:
%sql
-- REQUIREMENT 1: SQL SELECT/Keys & WHERE (Filter by status values)
SELECT order_id, customer_sk, order_status, total_sales_amount 
FROM fact_orders 
WHERE order_status = 'Delivered';

order_id,customer_sk,order_status,total_sales_amount
ORD_00001,28,Delivered,15000.0
ORD_00002,99,Delivered,5000.0
ORD_00003,55,Delivered,8000.0
ORD_00004,80,Delivered,5000.0
ORD_00005,10,Delivered,32000.0
ORD_00006,13,Delivered,10000.0
ORD_00008,52,Delivered,10000.0
ORD_00009,66,Delivered,12000.0
ORD_00011,16,Delivered,5000.0
ORD_00013,44,Delivered,32000.0


In [0]:
%sql
-- REQUIREMENT 2: GROUP BY + JOINS (Customer 360 & Revenue metrics evaluation)
SELECT 
    c.customer_name, 
    c.email,
    COUNT(f.order_id) as total_orders_placed,
    SUM(f.total_sales_amount) as total_revenue_yield
FROM dim_customer c
JOIN fact_orders f ON c.customer_sk = f.customer_sk
GROUP BY c.customer_name, c.email
ORDER BY total_revenue_yield DESC;

customer_name,email,total_orders_placed,total_revenue_yield
Sonya Foster,donaldmoore@example.com,8,701000.0
Sergio Vazquez,jennifersmith@example.org,7,682000.0
Matthew Tran,jeffrey32@example.net,5,597500.0
Michael Shannon,joseph91@example.net,6,571000.0
Dominique Sanders,kingsara@example.org,7,510000.0
Anthony Roberts,cheryldavidson@example.com,6,495000.0
Dr. Sydney Lopez,pthompson@example.com,6,489000.0
Aaron Martinez,luisgarcia@example.net,6,473000.0
Michael Mitchell,ashleyspence@example.org,6,464000.0
Anne Weaver,rhernandez@example.net,6,460500.0


In [0]:
%sql
-- REQUIREMENT 3: Conditional CASE Statements execution (Loyalty Segments Profiling)
SELECT 
    customer_sk, 
    COUNT(order_id) as total_transactions,
    CASE 
        WHEN COUNT(order_id) >= 5 THEN 'Elite Loyalty Tier'
        WHEN COUNT(order_id) BETWEEN 2 AND 4 THEN 'Regular Core Tier'
        ELSE 'Standard Entry Tier' 
    END AS customer_loyalty_segment
FROM fact_orders 
GROUP BY customer_sk;

customer_sk,total_transactions,customer_loyalty_segment
28,5,Elite Loyalty Tier
99,8,Elite Loyalty Tier
55,5,Elite Loyalty Tier
80,4,Regular Core Tier
10,8,Elite Loyalty Tier
13,4,Regular Core Tier
52,7,Elite Loyalty Tier
66,5,Elite Loyalty Tier
1,6,Elite Loyalty Tier
16,6,Elite Loyalty Tier


In [0]:
%sql
-- REQUIREMENT 4: Subqueries Isolation (Isolating Above-Average High Spending Orders)
SELECT order_id, customer_sk, total_sales_amount 
FROM fact_orders
WHERE total_sales_amount > (SELECT AVG(total_sales_amount) FROM fact_orders)
ORDER BY total_sales_amount DESC;

order_id,customer_sk,total_sales_amount
ORD_00347,38,360000.0
ORD_00282,100,360000.0
ORD_00251,10,285000.0
ORD_00037,88,285000.0
ORD_00327,95,240000.0
ORD_00174,92,240000.0
ORD_00376,71,240000.0
ORD_00046,51,240000.0
ORD_00089,21,240000.0
ORD_00353,19,240000.0


In [0]:
%sql
-- REQUIREMENT 5: CTEs & Window Functions combined (Rank Trending Products within Categories)
WITH TopTrendingProductsCTE AS (
    SELECT 
        p.product_name, 
        p.category, 
        SUM(f.quantity) as total_units_sold,
        DENSE_RANK() OVER (PARTITION BY p.category ORDER BY SUM(f.quantity) DESC) as sales_performance_rank
    FROM dim_product p
    JOIN fact_orders f ON p.product_sk = f.product_sk
    GROUP BY p.product_name, p.category
)
SELECT * FROM TopTrendingProductsCTE 
WHERE sales_performance_rank = 1;

product_name,category,total_units_sold,sales_performance_rank
Type-C Charging Hub,Accessories,57,1
Bluetooth Speaker,Audio,73,1
4K Ultra Monitor,Electronics,92,1
Smart Fitness Band,Wearables,57,1


In [0]:
import os

# Check raw data folder
print("--- Raw Files ---")
print("customers.csv exists:", os.path.exists("data/customers.csv"))
print("products.csv exists:", os.path.exists("data/products.csv"))
print("orders.csv exists:", os.path.exists("data/orders.csv"))

# Check pandas cleaned data folder
print("\n--- Cleaned Files ---")
print("customers_clean.csv exists:", os.path.exists("data/cleaned/customers_clean.csv"))
print("products_clean.csv exists:", os.path.exists("data/cleaned/products_clean.csv"))
print("orders_clean.csv exists:", os.path.exists("data/cleaned/orders_clean.csv"))

--- Raw Files ---
customers.csv exists: True
products.csv exists: True
orders.csv exists: True

--- Cleaned Files ---
customers_clean.csv exists: True
products_clean.csv exists: True
orders_clean.csv exists: True
